# 🧪 实验室 L1: 面板序列自相关与 BDM 45% 过度拒绝率沙盒
---
*实证经济学与计量模拟沙盒*

### 📖 实验室目标
1. **见证“学术灾难”的重现**：通过 Monte Carlo 模拟，直观感受不聚类时，随着序列自相关系数 $\rho$ 增加，名义 5% 假阳性率飙升至 ~40% 以上的退化过程。
2. **对比主流推断方案的拯救效果**：实时计算 `IID标准误`、`HC1稳健标准误` 及 `组内聚类标准误`（Clustered Standard Errors），验证它们在自相关下的真实拒绝率。
3. **自由调参，形成直觉**：通过交互滑块调节自相关系数 $\rho$、模拟次数、以及聚类组数 $G$，建立关于面板数据标准误设定层级的直觉。

## 1. 💡 物理机制与数理直觉

在双重差分 (DID) 与面板回归中，经典的 OLS 标准误假设残差是独立同分布 (iid) 的。然而，经济学中的许多变量都具有高度的**时间序列相关性 (Serial Correlation)**。比如，个体的今天受昨天影响（AR(1) 过程）：
$$\epsilon_{it} = \rho \epsilon_{i,t-1} + v_{it}$$

### 为什么不聚类会导致标准误被严重低估？
在 $\rho > 0$ 时，同一个体不同时期的残差包含重叠的信息。如果我们假装他们是互相独立的，相当于高估了数据中的**有效信息量（有效样本量）**，导致我们低估残差的真实方差，并高估 $t$ 统计量的绝对值，从而导致本不显著的政策效应被错误地判定为显著。

**Bertrand, Duflo, and Mullainathan (2004, QJE)** 指出，在这种设定下，即使真实政策效应为 0，传统的推断方法的假阳性率也会高达 **35% - 45%**！唯一的完美解决方案，是在政策实施的粗粒度层级（如省份级）进行聚类调整。

## 2. 🎛️ 交互式仿真引擎 (Simulation Engine)

运行下面的单元格以启动交互滑块。每次拖动滑块调整参数时，后台将实时运行蒙特卡洛模拟，并在下方瞬间更新假阳性率对比柱状图。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from scipy import stats

# 解决中文显示与负号渲染问题
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

def run_interactive_simulation(rho=0.8, n_sims=100, n_groups=50, n_units_per_group=10, n_periods=10):
    rng = np.random.default_rng(42)
    n_total_units = n_groups * n_units_per_group
    
    p_iid_list = []
    p_hc_list = []
    p_cl_list = []
    
    # 蒙特卡洛模拟核心循环
    for sim in range(n_sims):
        # 生成固定效应
        unit_fe = rng.normal(0, 1, n_total_units)
        time_fe = rng.normal(0, 1, n_periods)
        
        # 构造自回归误差项
        eps = np.zeros((n_total_units, n_periods))
        eps[:, 0] = rng.normal(0, 1, n_total_units)
        for t in range(1, n_periods):
            eps[:, t] = rho * eps[:, t - 1] + rng.normal(0, 1, n_total_units)
            
        # 分配省份组 ID
        group_ids = np.repeat(np.arange(n_groups), n_units_per_group)
        
        # 随机抽取 20% 的省份作为虚拟政策处理组
        treated_groups = rng.choice(n_groups, size=max(1, n_groups // 5), replace=False)
        is_treated_unit = np.isin(group_ids, treated_groups)
        
        # 政策矩阵 D (t >= 5 发生政策)
        D = np.zeros((n_total_units, n_periods))
        D[is_treated_unit, 5:] = 1
        
        # 观测值 Y (真实处理效应为 0)
        Y = unit_fe[:, None] + time_fe[None, :] + 0 * D + eps
        
        # 双向去均值得到 Within Panel Estimator 设定
        Y_dm = Y - Y.mean(axis=1, keepdims=True) - Y.mean(axis=0) + Y.mean()
        D_dm = D - D.mean(axis=1, keepdims=True) - D.mean(axis=0) + D.mean()
        
        y_flat = Y_dm.flatten()
        x_flat = D_dm.flatten()
        
        # OLS 估计量
        beta_hat = (x_flat * y_flat).sum() / (x_flat ** 2).sum()
        resid = y_flat - beta_hat * x_flat
        
        df_adj = n_total_units * n_periods - n_total_units - n_periods
        
        # 1. iid 标准误计算
        se_iid = np.sqrt((resid ** 2).sum() / df_adj / (x_flat ** 2).sum())
        t_iid = abs(beta_hat / se_iid)
        p_iid = 2 * (1 - stats.t.cdf(t_iid, df=df_adj))
        
        # 2. HC1 稳健标准误计算
        se_hc = np.sqrt(((x_flat * resid) ** 2).sum() / (x_flat ** 2).sum() ** 2 * (n_total_units * n_periods) / df_adj)
        t_hc = abs(beta_hat / se_hc)
        p_hc = 2 * (1 - stats.t.cdf(t_hc, df=df_adj))
        
        # 3. 组内聚类标准误计算 (聚类在省份 group 级别)
        x_reshaped = x_flat.reshape(n_groups, -1)
        r_reshaped = resid.reshape(n_groups, -1)
        group_scores = (x_reshaped * r_reshaped).sum(axis=1)
        se_cl = np.sqrt((group_scores ** 2).sum() / (x_flat ** 2).sum() ** 2 * n_groups / (n_groups - 1))
        t_cl = abs(beta_hat / se_cl)
        p_cl = 2 * (1 - stats.t.cdf(t_cl, df=n_groups - 1))
        
        p_iid_list.append(p_iid)
        p_hc_list.append(p_hc)
        p_cl_list.append(p_cl)
        
    # 计算实际拒绝率（%）
    rej_iid = (np.array(p_iid_list) < 0.05).mean() * 100
    rej_hc = (np.array(p_hc_list) < 0.05).mean() * 100
    rej_cl = (np.array(p_cl_list) < 0.05).mean() * 100
    
    # 绘图显示结果
    fig, ax = plt.subplots(figsize=(10, 6.5))
    bars = ax.bar(['IID 标准误', 'HC1 稳健标准误', '省份级聚类标准误'], [rej_iid, rej_hc, rej_cl],
                  color=['#AE0B2A', '#D9A441', '#003153'], edgecolor='black', width=0.55)
    
    ax.axhline(5, color='darkgreen', linestyle='--', linewidth=1.8, label='名义显著性水平 (5%)')
    ax.set_ylabel('名义 5% 时实际的拒绝率 (假阳性率 %)', fontsize=12, fontweight='bold')
    ax.set_ylim(0, max(rej_iid + rej_hc + 15, 20))
    ax.set_title(f'蒙特卡洛仿真实验：AR(1) 自相关系数 \u03c1 = {rho} 时的推断失效情况', fontsize=14, pad=15, fontweight='bold')
    
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.1f}%', (bar.get_x() + bar.get_width() / 2., height + 0.5),
                    ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontsize=11, fontweight='bold')
    
    ax.legend(fontsize=11)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

# 绑定滑块交互
widgets.interact(
    run_interactive_simulation,
    rho=widgets.FloatSlider(value=0.8, min=0.0, max=0.9, step=0.1, description='自相关系数 \u03c1'),
    n_sims=widgets.IntSlider(value=100, min=50, max=300, step=50, description='模拟次数'),
    n_groups=widgets.IntSlider(value=50, min=10, max=100, step=10, description='省份组数 G')
);

## 3. 🎯 学术启发与直觉透析

当我们拖动上面的滑块时，可以明显观察到三个核心学术直觉：

### 一、 误差自相关是低估标准误的物理根源
当滑块 **自相关系数 $\rho = 0.0$** 时，残差互相独立，IID、稳健标准误和聚类标准误得到的假阳性率都在 5% 线附近波动。这证明在数据没有时间序列相关性时，所有标准误都是有效的。

但是，当拖动滑块将 **$\rho$ 调大到 0.8** 时，IID 和 HC1 稳健标准误立刻崩塌，实际假阳性率瞬间**飙升至 35% 以上**。这意味着即使政策完全没有任何实际效果（真实效应为 0），我们也会有接近 40% 的概率判定该政策“在 5% 的统计水平上显著”。这正是 Bertrand 等人在 2004 年敲响的警钟。

### 二、 HC1 稳健标准误不能拯救时间序列相关性
很多同学认为“我的回归已经加了稳健标准误（在 Stata 中加了 `, r` 或 `, robust`，在 Python 中加了 `HC1`），结果就是稳健的”。
交互图表向我们展示了残酷的现实：**稳健标准误（黄柱）在自相关下的假阳性率依然极高，其表现几乎与 IID 标准误（红柱）完全一样，没有任何纠偏能力。**
这是因为，所谓的稳健标准误（Huber-White）**只纠正异方差**，在其方差-协方差矩阵（VCE）中，所有非对角线元素依然被强制设为了 0，即完全无视了个体内部的前后序列相关性。

### 三、 组内聚类（Clustered SE）的本质是方差-协方差对角块的“完全松绑”
当我们在省份级（`group`）进行聚类调整时（蓝柱），无论 $\rho$ 增加到多少，假阳性率都会**稳稳地被拽回 5% 名义显著性水平附近**。这是因为聚类标准误允许同一个省份内的个体误差在时间上存在任意形式的序列相关、任意形式的异方差。这相当于将方差-协方差矩阵中代表同一省份不同时期的“非对角块”完全松绑释放，允许它们不为 0。因此，它能够彻底清算自相关带来的推断低估问题。